## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 📋 Loan Underwriting Agent with Azure AI Search

This notebook demonstrates how to use **Azure AI Search with Agentic Mode** for RAG (Retrieval Augmented Generation) in a **Loan Underwriting** scenario.

## Features Covered:
- Setting up `AzureAISearchContextProvider` with **agentic mode**
- Creating a Knowledge Base for underwriting policy documents
- Multi-hop reasoning across underwriting guidelines and criteria
- Intelligent query planning for complex eligibility questions

### Industry Use Case: Loan Underwriting & Risk Assessment

Our agent will help underwriters and loan officers:
- Review underwriting guidelines and approval criteria
- Analyze debt-to-income (DTI) and loan-to-value (LTV) requirements
- Research documentation requirements for different loan types
- Synthesize eligibility criteria across multiple loan products

### ⚠️ Important Disclaimer ⚠️
> **This notebook is for educational purposes only. All underwriting criteria is sample data for training purposes. Always follow your institution's official underwriting guidelines and regulatory requirements.**

### 🔍 Why Agentic Mode?

| Feature | Agentic Mode | Semantic Mode |
|---------|-------------|---------------|
| Query Planning | ✅ Multi-hop reasoning | ❌ Single query |
| Knowledge Bases | ✅ Required | ❌ Uses index directly |
| Accuracy | Higher (36% improvement) | Good for simple queries |
| Speed | Slightly slower | Faster |
| Best For | Complex questions | Simple lookups |

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Search Service**: With a search index or knowledge base
2. **Microsoft Foundry Project**: With a deployed model (gpt-4o recommended)
3. **Authentication**: Azure CLI installed and authenticated
4. **Environment Variables** in root `.env` file:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
   - `AZURE_AI_SEARCH_ENDPOINT`
   - `AZURE_AI_SEARCH_API_KEY` (optional if using managed identity)
   - `AZURE_SEARCH_INDEX_NAME` (for auto-creating Knowledge Base)
   - `AZURE_OPENAI_ENDPOINT` (required for agentic mode with index)

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

## Import Libraries

Import the required libraries for Azure AI Search context provider:

In [2]:
# Copyright (c) Microsoft. All rights reserved.
import os
import sys
from importlib.metadata import version
from pathlib import Path

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

try:
    from agent_framework.azure import AzureAISearchContextProvider
    SEARCH_PROVIDER_AVAILABLE = True
except ModuleNotFoundError:
    SEARCH_PROVIDER_AVAILABLE = False
    AzureAISearchContextProvider = None
    print("⚠️ Azure AI Search context provider is unavailable in this environment. "
          "Install the optional connector with 'pip install agent-framework-azure-ai --pre' to enable the provider flow.")
except ImportError:
    SEARCH_PROVIDER_AVAILABLE = False
    AzureAISearchContextProvider = None
    print("⚠️ The installed Azure AI Search connector is not compatible with the current agent framework core. "
          "The search-provider flow is disabled.")

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

# Verify environment setup
required_vars = [
    'AI_FOUNDRY_PROJECT_ENDPOINT',
    'AZURE_AI_MODEL_DEPLOYMENT_NAME',
    'AZURE_AI_SEARCH_ENDPOINT',
]

missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    print(f"⚠️ Missing environment variables: {missing}")
    print("Please configure these in your root .env file")
else:
    print("🔧 Environment Configuration:")
    print("✅ Project Endpoint: configured (value hidden)")
    print(f"✅ Model Deployment: {os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')}")
    print(f"✅ Search Endpoint: {os.getenv('AZURE_AI_SEARCH_ENDPOINT')}")
    print(f"✅ Search Index: {os.getenv('AZURE_SEARCH_INDEX_NAME', 'Not set - will need KB name')}")

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print({package: version(package) for package in ("agent-framework-core", "agent-framework-azure-ai")})


🔧 Environment Configuration:
✅ Project Endpoint: configured (value hidden)
✅ Model Deployment: gpt-4.1
✅ Search Endpoint: https://demofoundryaisearch.search.windows.net
✅ Search Index: underwriting-index
Kernel: c:\src\agentic-ai-immersion\.venv\Scripts\python.exe
Python: 3.13.15
{'agent-framework-core': '1.17.0', 'agent-framework-azure-ai': '1.0.0rc6'}


## Configuration 📋

Set up the configuration for Azure AI Search and the agent. We'll use agentic mode for intelligent multi-hop retrieval.

In [3]:
# Azure AI Search configuration
SEARCH_ENDPOINT = os.environ["AZURE_AI_SEARCH_ENDPOINT"]
SEARCH_API_KEY = os.environ.get("AZURE_AI_SEARCH_API_KEY")  # Optional if using managed identity

# Microsoft Foundry configuration
PROJECT_ENDPOINT = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

# For agentic mode, we need either:
# Option 1: An existing Knowledge Base name
KNOWLEDGE_BASE_NAME = os.environ.get("AZURE_SEARCH_KNOWLEDGE_BASE_NAME")

# Option 2: Auto-create KB from an index (requires OpenAI endpoint)
INDEX_NAME = os.environ.get("AZURE_SEARCH_INDEX_NAME")
AZURE_OPENAI_RESOURCE_URL = os.environ.get("AZURE_OPENAI_ENDPOINT")

print("📊 Search Configuration:")
print(f"  Endpoint: {SEARCH_ENDPOINT}")
print(f"  API Key: {'Configured' if SEARCH_API_KEY else 'Using managed identity'}")
print(f"  Knowledge Base: {KNOWLEDGE_BASE_NAME or 'Will auto-create from index'}")
print(f"  Index Name: {INDEX_NAME or 'Not configured'}")
print(f"  OpenAI Resource: {AZURE_OPENAI_RESOURCE_URL or 'Not configured'}")

📊 Search Configuration:
  Endpoint: https://demofoundryaisearch.search.windows.net
  API Key: Using managed identity
  Knowledge Base: platform-kb
  Index Name: underwriting-index
  OpenAI Resource: https://demopocaifoundry.openai.azure.com/openai/v1


## Define Underwriting Research Queries 📋

These sample queries demonstrate the loan underwriting use case. The agentic mode excels at:
- Complex multi-hop queries requiring synthesis across policy documents
- Questions that need reasoning about eligibility criteria relationships
- Analysis requests that span multiple loan product guidelines

## 🔧 Create Index and Upload Sample Data

Run this cell **once** to create the `underwriting-index` and upload sample loan underwriting documents. This cell will:
1. Create the Azure AI Search index with the proper schema
2. Upload sample underwriting policy documents
3. Verify the data was uploaded successfully

> **Note**: Skip this cell if the index already exists with data.

In [ ]:
import time
import os
from openai import AzureOpenAI, NotFoundError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential, get_bearer_token_provider

index_name = "underwriting-index"
knowledge_source_name = f"{index_name}-source"
knowledge_base_name = f"{index_name}-kb"
embedding_deployment = (
    os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
    or os.environ.get("AZURE_AI_EMBEDDING_DEPLOYMENT")
    or "text-embedding-3-large"
)

azure_openai_endpoint = (AZURE_OPENAI_RESOURCE_URL or "").rstrip("/")
if not azure_openai_endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT is not configured. Set it in your .env file before running the index setup."
    )

cli_credential = AzureCliCredential()
credential = AzureKeyCredential(SEARCH_API_KEY) if SEARCH_API_KEY else cli_credential
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

aoai_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
if aoai_api_key:
    aoai_client = AzureOpenAI(
        azure_endpoint=azure_openai_endpoint,
        api_key=aoai_api_key,
        api_version="2024-06-01",
    )
else:
    token_provider = get_bearer_token_provider(
        cli_credential,
        "https://cognitiveservices.azure.com/.default",
    )
    aoai_client = AzureOpenAI(
        azure_endpoint=azure_openai_endpoint,
        azure_ad_token_provider=token_provider,
        api_version="2024-06-01",
    )

try:
    aoai_client.embeddings.create(input=["test"], model=embedding_deployment)
    print(f"✅ Azure OpenAI embedding deployment '{embedding_deployment}' is reachable")
except NotFoundError as exc:
    raise RuntimeError(
        f"The Azure OpenAI embedding deployment '{embedding_deployment}' was not found on '{azure_openai_endpoint}'. "
        "Create the deployment in Azure OpenAI Studio or set AZURE_OPENAI_EMBEDDING_DEPLOYMENT to the correct deployment name."
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Azure OpenAI embeddings could not be initialized. Check the endpoint, authentication, and deployment name. Details: {exc}"
    ) from exc

UNDERWRITING_DOCUMENTS = [
    {
        "id": "uw-001",
        "title": "Conventional Loan Underwriting Guidelines",
        "content": """Conventional Loan Underwriting Criteria:\n\n1. Credit Score Requirements:\n- Minimum credit score: 620 for most conventional loans\n- For best rates: 740+ recommended\n\n2. Debt-to-Income (DTI) Ratio:\n- Maximum front-end DTI: 28%\n- Maximum back-end DTI: 36-43%\n""",
    },
    {
        "id": "uw-002",
        "title": "FHA Loan Underwriting Guidelines",
        "content": """FHA Loan Underwriting Criteria:\n\n1. Credit Score Requirements:\n- Minimum 580 for 3.5% down\n- 500-579 requires 10% down\n\n2. Debt-to-Income (DTI) Ratio:\n- Front-end DTI max 31%\n- Back-end DTI max 43%\n""",
    },
    {
        "id": "uw-003",
        "title": "Income Verification Requirements",
        "content": """Income Verification Documentation Requirements:\n\n1. W-2 Employees:\n- Last 2 years W-2 forms\n- Most recent 30 days pay stubs\n\n2. Self-Employed Borrowers:\n- Last 2 years tax returns\n- Year-to-date profit and loss statement\n""",
    },
]

print("🧹 Cleaning up existing resources...")
print("=" * 60)

try:
    index_client.get_knowledge_base(knowledge_base_name)
    print(f"🗑️ Deleting knowledge base '{knowledge_base_name}'...")
    index_client.delete_knowledge_base(knowledge_base_name)
    time.sleep(2)
    print("   ✅ Knowledge base deleted")
except ResourceNotFoundError:
    print(f"   ℹ️ Knowledge base '{knowledge_base_name}' not found (OK)")
except Exception as exc:
    print(f"   ⚠️ Could not delete knowledge base: {exc}")

try:
    index_client.get_knowledge_source(knowledge_source_name)
    print(f"🗑️ Deleting knowledge source '{knowledge_source_name}'...")
    index_client.delete_knowledge_source(knowledge_source_name)
    time.sleep(1)
    print("   ✅ Knowledge source deleted")
except ResourceNotFoundError:
    print(f"   ℹ️ Knowledge source '{knowledge_source_name}' not found (OK)")
except Exception as exc:
    print(f"   ⚠️ Could not delete knowledge source: {exc}")

print(f"\n{'=' * 60}")
print("✅ Cleanup complete - ready to create index")


def create_search_index(index_name):
    """Create a vector search index with hybrid search capabilities."""
    try:
        index_client.get_index(index_name)
        print(f"   🗑️ Deleting existing index '{index_name}'...")
        index_client.delete_index(index_name)
        time.sleep(2)
    except ResourceNotFoundError:
        pass

    vector_search = VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
        profiles=[
            VectorSearchProfile(
                name="vector-profile",
                algorithm_configuration_name="hnsw-config",
                vectorizer_name="openai-vectorizer",
            )
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="openai-vectorizer",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=azure_openai_endpoint,
                    deployment_name=embedding_deployment,
                    model_name=embedding_deployment,
                ),
            )
        ],
    )

    semantic_search = SemanticSearch(
        default_configuration_name="semantic-config",
        configurations=[
            SemanticConfiguration(
                name="semantic-config",
                prioritized_fields=SemanticPrioritizedFields(
                    content_fields=[SemanticField(field_name="content")]
                ),
            )
        ],
    )

    fields = [
        SearchField(name="id", type=SearchFieldDataType.String, key=True),
        SearchField(name="title", type=SearchFieldDataType.String, searchable=True),
        SearchField(name="content", type=SearchFieldDataType.String, searchable=True),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=3072,
            vector_search_profile_name="vector-profile",
        ),
    ]

    index = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search,
        semantic_search=semantic_search,
    )
    index_client.create_or_update_index(index)


def generate_embeddings(texts, batch_size=10):
    """Generate embeddings using Azure OpenAI."""
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        try:
            response = aoai_client.embeddings.create(input=batch, model=embedding_deployment)
        except NotFoundError as exc:
            raise RuntimeError(
                f"The Azure OpenAI embedding deployment '{embedding_deployment}' was not found or is inaccessible. "
                "Verify the deployment name and endpoint in your .env file."
            ) from exc
        embeddings.extend([item.embedding for item in response.data])
    return embeddings


def upload_documents_to_index(index_name, documents):
    """Upload documents with embeddings to search index."""
    search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=index_name, credential=credential)
    texts = [doc["content"] for doc in documents]
    embeddings = generate_embeddings(texts)
    for i, doc in enumerate(documents):
        doc["content_vector"] = embeddings[i]
    search_client.upload_documents(documents)
    return len(documents)


print("\n🏗️ Creating Azure AI Search Index with Vector Search...")
print("=" * 60)
print(f"\n📚 Creating: {index_name}")
create_search_index(index_name)
print("   ✅ Index created successfully")

print("\n🧠 Generating embeddings and uploading documents...")
print("=" * 60)
print(f"\n📤 Processing: {index_name}")
print(f"   ⏳ Generating {len(UNDERWRITING_DOCUMENTS)} embeddings...")
count = upload_documents_to_index(index_name, UNDERWRITING_DOCUMENTS)
print(f"   ✅ Uploaded {count} documents")

time.sleep(2)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=index_name, credential=credential)
doc_count = search_client.get_document_count()

print(f"\n{'=' * 60}")
print("✅ Index setup complete!")
print(f"   • Index name: {index_name}")
print(f"   • Document count: {doc_count}")
print(f"   • Embedding model: {embedding_deployment}")
print("   • Vector dimensions: 3072")
print("   • Semantic config: semantic-config")

In [5]:
# FSI Loan Underwriting sample queries
UNDERWRITING_QUERIES = [
    # Policy overview query
    "What are the key underwriting criteria for mortgage loan approval?",
    
    # Risk assessment query  
    "What debt-to-income (DTI) ratios are acceptable for different loan types?",
    
    # Documentation requirements
    "What documentation is required for income verification in loan applications?",
    
    # Eligibility criteria
    "What credit score requirements apply to conventional vs FHA loans?",
]

print("📋 Sample Loan Underwriting Research Queries:")
for i, query in enumerate(UNDERWRITING_QUERIES, 1):
    print(f"  {i}. {query[:80]}{'...' if len(query) > 80 else ''}")

📋 Sample Loan Underwriting Research Queries:
  1. What are the key underwriting criteria for mortgage loan approval?
  2. What debt-to-income (DTI) ratios are acceptable for different loan types?
  3. What documentation is required for income verification in loan applications?
  4. What credit score requirements apply to conventional vs FHA loans?


## Create Search Context Provider 🔍

The `AzureAISearchContextProvider` in **agentic mode** uses Knowledge Bases for intelligent query planning and multi-hop retrieval.

**Key Parameters:**
- `mode="agentic"`: Enables multi-hop reasoning
- `knowledge_base_name`: Use existing KB (recommended)
- `index_name` + `azure_openai_resource_url`: Auto-create KB from index
- `knowledge_base_output_mode`: `"extractive_data"` or `"answer_synthesis"`
- `retrieval_reasoning_effort`: `"minimal"`, `"low"`, or `"medium"`

> **Note**: For this notebook, we'll use the `underwriting-index` which contains loan underwriting policies, eligibility criteria, and documentation requirements.

In [4]:
def create_search_provider():
    """Create Azure AI Search context provider with agentic mode when available."""
    if not SEARCH_PROVIDER_AVAILABLE:
        raise RuntimeError(
            "Azure AI Search context provider is unavailable in this environment. "
            "The notebook will continue with the index setup and a fallback agent pattern."
        )

    print("🔍 Creating Azure AI Search Context Provider (Agentic Mode)...")
    print("   This mode uses Knowledge Bases for intelligent query planning.\n")

    # Configure based on available settings
    if KNOWLEDGE_BASE_NAME:
        print(f"✅ Using existing Knowledge Base: {KNOWLEDGE_BASE_NAME}")
        return AzureAISearchContextProvider(
            endpoint=SEARCH_ENDPOINT,
            api_key=SEARCH_API_KEY,
            credential=AzureCliCredential() if not SEARCH_API_KEY else None,
            mode="agentic",
            knowledge_base_name=KNOWLEDGE_BASE_NAME,
            knowledge_base_output_mode="extractive_data",
            retrieval_reasoning_effort="minimal",
        )
    elif INDEX_NAME and AZURE_OPENAI_RESOURCE_URL:
        print(f"✅ Auto-creating Knowledge Base from index: {INDEX_NAME}")
        return AzureAISearchContextProvider(
            endpoint=SEARCH_ENDPOINT,
            index_name=INDEX_NAME,
            api_key=SEARCH_API_KEY,
            credential=AzureCliCredential() if not SEARCH_API_KEY else None,
            mode="agentic",
            azure_openai_resource_url=AZURE_OPENAI_RESOURCE_URL,
            model=MODEL_DEPLOYMENT,
            knowledge_base_output_mode="extractive_data",
            retrieval_reasoning_effort="minimal",
            top_k=5,
        )
    else:
        raise ValueError(
            "Configure either AZURE_SEARCH_KNOWLEDGE_BASE_NAME or both "
            "AZURE_SEARCH_INDEX_NAME and AZURE_OPENAI_ENDPOINT in your .env file"
        )

# Test provider creation
try:
    test_provider = create_search_provider()
    print("\n✅ Search provider configuration validated!")
except (RuntimeError, ValueError) as e:
    print(f"\n⚠️ Search provider setup skipped: {e}")


🔍 Creating Azure AI Search Context Provider (Agentic Mode)...
   This mode uses Knowledge Bases for intelligent query planning.

✅ Using existing Knowledge Base: platform-kb

✅ Search provider configuration validated!


## Underwriting Agent Instructions 📋

We define comprehensive instructions for our loan underwriting agent that:
- Specializes in underwriting policy analysis and eligibility assessment
- Uses retrieved context for accurate criteria interpretation
- Includes appropriate regulatory disclaimers

In [6]:
AGENT_INSTRUCTIONS = """
You are a Loan Underwriting Research Assistant with expertise in mortgage and lending guidelines.

## Your Capabilities:
- Analyze underwriting criteria including DTI ratios, LTV limits, and credit requirements
- Explain documentation requirements for various loan types
- Compare eligibility criteria across conventional, FHA, VA, and jumbo loans
- Synthesize information across multiple underwriting policy documents

## Guidelines:
1. **Use Context**: Base your answers on the retrieved underwriting policy information from the knowledge base
2. **Be Precise**: Cite specific ratios, thresholds, or requirements from the sources
3. **Acknowledge Gaps**: If underwriting criteria is not available, clearly state this
4. **Professional Tone**: Maintain a helpful, analytical communication style

## Required Disclaimers:
- Always include: "This is for informational and training purposes only."
- Recommend consulting official underwriting guidelines for actual loan decisions
- Note that underwriting criteria may vary by institution and are subject to change

## Response Format:
- Start with a direct answer to the question
- Provide supporting details from retrieved underwriting policies
- End with relevant caveats or recommendations
"""

print("📝 Agent Instructions Configured")
print(f"   Length: {len(AGENT_INSTRUCTIONS)} characters")

📝 Agent Instructions Configured
   Length: 1242 characters


## Run the Underwriting Research Agent 🚀

Now we create and run our loan underwriting agent with the Azure AI Search context provider in agentic mode.

In [7]:
async def run_underwriting_agent():
    """Run the loan underwriting research agent with agentic search."""
    
    print("=" * 60)
    print("📋 Loan Underwriting Research Agent")
    print("   Using Azure AI Search with Agentic Mode")
    print("=" * 60 + "\n")
    
    # Create the search context provider
    search_provider = create_search_provider()
    
    credential = AzureCliCredential()
    async with search_provider:
        client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        
        agent = Agent(
            client=client,
            name="underwriting-agent",
            instructions=AGENT_INSTRUCTIONS,
            context_providers=[search_provider],
        )
        
        print(f"✅ Agent created: {agent.name}")
        print(f"🔍 Search mode: Agentic (multi-hop reasoning)")
        print("\n" + "-" * 60 + "\n")
        
        # Process each underwriting query
        for i, query in enumerate(UNDERWRITING_QUERIES, 1):
            print(f"📊 Query {i}: {query}")
            print("\n💬 Agent Response:")
            
            # Stream the response for better UX
            async for chunk in agent.run(query, stream=True):
                if chunk.text:
                    print(chunk.text, end="", flush=True)
            
            print("\n\n" + "-" * 60 + "\n")
        
        print("✅ Underwriting analysis complete!")

## Execute the Agent 🎯

Run the loan underwriting research agent. The agent uses the `underwriting-index` which contains underwriting policies, eligibility criteria, DTI/LTV requirements, and documentation guidelines.

In [8]:
# Run the loan underwriting research agent
await run_underwriting_agent()

📋 Loan Underwriting Research Agent
   Using Azure AI Search with Agentic Mode

🔍 Creating Azure AI Search Context Provider (Agentic Mode)...
   This mode uses Knowledge Bases for intelligent query planning.

✅ Using existing Knowledge Base: platform-kb
✅ Agent created: underwriting-agent
🔍 Search mode: Agentic (multi-hop reasoning)

------------------------------------------------------------

📊 Query 1: What are the key underwriting criteria for mortgage loan approval?

💬 Agent Response:
The key underwriting criteria for mortgage loan approval typically include the following factors:

1. **Debt-to-Income (DTI) Ratio**:  
   Underwriting policies generally require a DTI ratio below a specified threshold. For conventional loans, the maximum DTI is often 43%, though some programs may allow ratios up to 50% with strong compensating factors. FHA loans typically allow DTI up to 43%, but can go higher (up to 57%) with manual underwriting and additional documentation.

2. **Loan-to-Value (LTV

## Interactive Query Mode 💬

Use this cell for custom underwriting research queries:

In [9]:
async def ask_underwriting_question(question: str):
    """Ask a custom loan underwriting research question."""
    
    search_provider = create_search_provider()
    
    credential = AzureCliCredential()
    async with search_provider:
        client = FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        )
        
        agent = Agent(
            client=client,
            name="underwriting-advisor",
            instructions=AGENT_INSTRUCTIONS,
            context_providers=[search_provider],
        )
        
        print(f"🤔 Question: {question}\n")
        print("💬 Response:")
        
        async for chunk in agent.run(question, stream=True):
            if chunk.text:
                print(chunk.text, end="", flush=True)
        print("\n")

# Example: Ask a custom question
# await ask_underwriting_question("What are the maximum LTV ratios for investment property loans?")

## Key Takeaways 📚

### Azure AI Search Context Provider

```python
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework.azure import AzureAISearchContextProvider

# Using existing Knowledge Base (recommended)
search_provider = AzureAISearchContextProvider(
    endpoint=search_endpoint,
    api_key=api_key,  # Optional with managed identity
    mode="agentic",
    knowledge_base_name="my-knowledge-base",
    knowledge_base_output_mode="extractive_data",
    retrieval_reasoning_effort="minimal",
)

# Create client and agent
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=model_deployment,
    credential=credential,
)
agent = Agent(
    client=client,
    instructions="Your instructions",
    context_providers=[search_provider],
)
```

### Agentic vs Semantic Mode

| Aspect | Agentic Mode | Semantic Mode |
|--------|-------------|---------------|
| **Query Planning** | Multi-hop reasoning | Single query |
| **Accuracy** | ~36% improvement | Good for simple queries |
| **Speed** | Slightly slower | Faster |
| **Token Usage** | Higher | Lower |
| **Best For** | Complex analysis | Simple lookups |

### FSI Best Practices for Underwriting

1. **Always Include Disclaimers**: Underwriting decisions require official policy adherence
2. **Use Extractive Mode**: For auditability, use `knowledge_base_output_mode="extractive_data"`
3. **Minimal Reasoning**: Start with `retrieval_reasoning_effort="minimal"` for faster responses
4. **Source Attribution**: Ensure responses cite the source policy documents for compliance

### Migration Note

| Old (Deprecated) | New |
|---|---|
| `AzureAIAgentClient` | `FoundryChatClient` |
| `ChatAgent(chat_client=..., context_provider=...)` | `Agent(client=..., context_providers=[...])` |
| `agent.run_stream(query)` | `agent.run(query, stream=True)` |

### Environment Variables Needed

```env
# Required
AI_FOUNDRY_PROJECT_ENDPOINT=https://your-project.services.ai.azure.com/...
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o
AZURE_AI_SEARCH_ENDPOINT=https://your-search.search.windows.net

# Option 1: Existing Knowledge Base
AZURE_SEARCH_KNOWLEDGE_BASE_NAME=my-knowledge-base

# Option 2: Auto-create from index
AZURE_SEARCH_INDEX_NAME=underwriting-index
AZURE_OPENAI_ENDPOINT=https://your-openai.openai.azure.com/

# Optional
AZURE_AI_SEARCH_API_KEY=your-api-key  # If not using managed identity
```

⚠️ **Disclaimer**: This notebook is for educational and training purposes. All underwriting scenarios are simulated. Always follow your institution's official underwriting guidelines and regulatory requirements for actual loan decisions.